# LinearFold
<!-- SPDX-License-Identifier: GPL-3.0-only -->

Adapted from `sinc-lab/lncRNA-folding`.
Modified by Jingwen Liu, 2026, for full-length viral RNA benchmarking.

In [ ]:
import os
import pandas as pd
import time
from pathlib import Path

In [ ]:
method_name = "LinearFold"
base = Path.cwd()

In [ ]:
# Change to RNA_Analysis_Tools directory for installation
os.chdir('../tools')
print(f"Current directory: {Path.cwd()}")

# Clone LinearFold if not already present
if not Path('LinearFold').exists():
    !git clone https://github.com/LinearFold/LinearFold.git
else:
    print("LinearFold already exists")

In [ ]:
# Build LinearFold
os.chdir("LinearFold")
print(f"Building LinearFold in: {Path.cwd()}")

# Check if already built
if not Path('bin/linearfold_v').exists():
    !make
    print("LinearFold built successfully")
else:
    print("LinearFold already built")

# Return to methods directory
os.chdir('../../methods')
print(f"Returned to: {Path.cwd()}")

In [ ]:
def read_virus_fasta(path: str):
    lines = [ln.strip() for ln in open(path, 'r').read().splitlines() if ln.strip() != '']
    records = []
    for i in range(0, len(lines), 3):
        header, seq, struct = lines[i], lines[i+1], lines[i+2]
        name = header[1:].strip()
        records.append((name, seq.strip(), struct.strip()))
    df = pd.DataFrame(records, columns=['name','sequence','structure']).set_index('name')
    return df

viruses = read_virus_fasta('../data/viruses.fasta')

selected_virus_keys = None

if selected_virus_keys is None:
    virus_ids = list(viruses.index)
else:
    tmp = []
    for k in selected_virus_keys:
        if isinstance(k, int):
            tmp.append(viruses.index[k])
        else:
            tmp.append(str(k))
    virus_ids = tmp

In [ ]:
# Set path to LinearFold executable
linearfold_exe = base.parent / 'tools' / 'LinearFold' / 'bin' / 'linearfold_v'
print(f"LinearFold executable: {linearfold_exe}")
print(f"Executable exists: {linearfold_exe.exists()}")

def run_folding(name, seq):
    tmp_fasta = f'LinearFold_tmp_{name}.fasta'
    tmp_dot = f'LinearFold_tmp_{name}.dot'
    clean_dot = f'LinearFold_clean_tmp_{name}.dot'
    with open(tmp_fasta, 'w') as ofile:
        ofile.write(f'>{name}\n{seq}\n')
    os.system(f'cat {tmp_fasta} | {linearfold_exe} > {tmp_dot}')
    in_lines = open(tmp_dot, 'r').readlines()
    with open(clean_dot, 'w') as out_file:
        for line in in_lines:
            if '>' in line:
                out_file.write(line.split(':')[1].strip() + '\n')
            else:
                out_file.write(line)
    return clean_dot, [tmp_fasta, tmp_dot, clean_dot]

In [ ]:
out_dir = Path.cwd().parent / 'prediction'
os.makedirs(out_dir, exist_ok=True)
out_fasta_path = out_dir / (method_name + ".fasta")
if os.path.exists(out_fasta_path): os.remove(out_fasta_path)

print(f"{' ':3}\t{'virus':<20}\t{'len':<5}\t{'time'}")
for i, vid in enumerate(virus_ids):
    start_time = time.time()
    seq = viruses.loc[vid]['sequence']
    print(f"{i+1:3d}/{len(virus_ids)}\t{vid:<20}\t{len(seq):<5}\t", end='', flush=True)
    dot_file_name, temp_files = run_folding(vid, seq)
    os.system('cat ' + dot_file_name + ' >> ' + str(out_fasta_path))
    elapsed_time = time.time() - start_time
    print(f"{elapsed_time: .1f} s")
    for temp_file in temp_files:
        if os.path.exists(temp_file):
            os.remove(temp_file)

print(f"\nProcessing complete. Results saved to {out_fasta_path.name}")